In [29]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

In [30]:
SEED = 42

TARGET_COL = "Total Sales"

df = pd.read_csv("Chevy_US_Data.csv")

df.dropna(subset=[TARGET_COL], inplace=True)

y = df[TARGET_COL].to_numpy(dtype=np.float64)

X_df = df.drop(columns=[TARGET_COL])



test_size = 0.15
val_size = 0.15


X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=test_size, random_state=SEED)

val_fraction = val_size / (1.0 - test_size)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=val_fraction, random_state=SEED)



from pandas.api.types import is_numeric_dtype


numeric_features = [c for c in X_df.columns if is_numeric_dtype(X_df[c])]
categorical_features = [c for c in X_df.columns if c not in numeric_features]


# Build Pipeline for numeric features

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ('scaler', StandardScaler())
])


# Build Pipeline for categorical features

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])



# Build Column Transformer

pre = ColumnTransformer(transformers=[
    ("num", num_pipe, numeric_features),
    ("cat", cat_pipe, categorical_features)
])


#X_train_p = pre.fit_transform(X_train)
#X_val_p = pre.transform(X_val)
#X_test_p = pre.transform(X_test)

C_values = np.logspace(-2, 3, 30)
gamma_values = np.logspace(-4, 1, 30)


In [31]:
linear_svm_pipe = Pipeline(steps=[
    ("preprocess", pre),
    ("model", SVR())
])



param_dist = {
    "model__kernel": ["linear"],
    "model__C": C_values,
    "model__gamma": gamma_values,
    "model__degree": [1],
    "model__coef0": [0.0]
}


search = RandomizedSearchCV(
    estimator=linear_svm_pipe,
    param_distributions=param_dist,
    n_iter=3,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    random_state=SEED,
    verbose=1,
)




search.fit(X_train, y_train)

best_model = search.best_estimator_

y_val_pred = best_model.predict(X_val)



rmse = mean_squared_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print("Validation RMSE:", rmse)
print("Validation R^2:", r2)




# Finally, evaluate on test set


y_test_pred = best_model.predict(X_test)


rmse = mean_squared_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)


print("Test RMSE:", rmse)
print("Test R^2:", r2)

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Validation RMSE: 86556992980.87059
Validation R^2: -0.9932982359219025
Test RMSE: 66028622106.85126
Test R^2: 0.6538817631063272
